# Audit Tata Kelola Sistem Informasi Akademik (SIAKAD) Menggunakan COBIT 5
**Tugas 2 Audit Sistem Informasi (AUSI) - Semester Ganjil 2026**

Notebook ini mengimplementasikan perhitungan *Process Capability Level* (skala 0–5), analisis kesenjangan (*Gap Analysis* antara skor kondisi saat ini/*As-Is* dan target/*To-Be*), pemeringkatan prioritas perbaikan, serta visualisasi hasil audit berdasarkan dataset kuesioner `data_kuesioner_siakad_cobit5.csv`.

## Tahap 1: Import Library dan Memuat Dataset Kuesioner
Membaca data hasil penilaian kuesioner dari 4 responden internal pengelola SIAKAD (*Head of IT*, *System Administrator*, *Developer*, dan *Helpdesk*).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Membaca dataset kuesioner dari file CSV
df = pd.read_csv("data_kuesioner_siakad_cobit5.csv")
df

## Tahap 2: Perhitungan Skor As-Is, Capability Level, Gap, dan Rekomendasi
Menghitung rata-rata nilai dari keempat responden sebagai skor kondisi saat ini (`Skor_As_Is`), mengonversi nilai ke tingkat kapabilitas COBIT 5 (*Level 0* hingga *Level 5*), menghitung selisih terhadap target (`Gap = Target_To_Be - Skor_As_Is`), dan mengurutkan prioritas perbaikan.

In [ ]:
def tentukan_level(skor):
    level = int(np.floor(skor))
    label = {
        0: "Level 0 - Incomplete",
        1: "Level 1 - Performed",
        2: "Level 2 - Managed",
        3: "Level 3 - Established",
        4: "Level 4 - Predictable",
        5: "Level 5 - Optimizing"
    }
    return label.get(level, "Level 0 - Incomplete")

def buat_rekomendasi(gap):
    if gap <= 0:
        return "Target tercapai. Pertahankan kontrol dan lakukan evaluasi berkala."
    elif gap <= 1.0:
        return "Gap minor. Perkuat standarisasi SOP dan kelengkapan dokumentasi."
    else:
        return "Gap signifikan. Prioritaskan perbaikan tata kelola dan sumber daya TI."

# Identifikasi otomatis seluruh kolom penilaian auditor
kolom_auditor = [col for col in df.columns if col.startswith("Auditor_")]

# Kalkulasi metrik COBIT 5
df["Skor_As_Is"] = df[kolom_auditor].mean(axis=1).round(2)
df["Capability_Level"] = df["Skor_As_Is"].apply(tentukan_level)
df["Gap"] = (df["Target_To_Be"] - df["Skor_As_Is"]).round(2)
df["Rekomendasi"] = df["Gap"].apply(buat_rekomendasi)

# Urutkan dari Gap terbesar sebagai prioritas utama perbaikan
df_hasil = df.sort_values(by="Gap", ascending=False).reset_index(drop=True)
df_hasil.insert(0, "Prioritas", df_hasil.index + 1)

print(f"Rata-Rata Skor As-Is Keseluruhan : {df['Skor_As_Is'].mean():.2f}")
print(f"Rata-Rata Target To-Be           : {df['Target_To_Be'].mean():.2f}")
print(f"Rata-Rata Gap Keseluruhan        : {df['Gap'].mean():.2f}\n")

df_hasil[["Prioritas", "Domain", "Kode Proses", "Nama Proses", "Skor_As_Is", "Target_To_Be", "Gap", "Capability_Level", "Rekomendasi"]]

## Tahap 3: Visualisasi Hasil Audit (As-Is vs To-Be & Peringkat Gap)
Menampilkan dua grafik utama untuk dokumentasi laporan: perbandingan skor kapabilitas saat ini dengan target yang diharapkan, serta peringkat nilai kesenjangan (*gap*) tiap proses.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grafik 1: Perbandingan Skor As-Is vs Target To-Be
x = np.arange(len(df["Kode Proses"]))
width = 0.35
bar1 = ax1.bar(x - width/2, df["Skor_As_Is"], width, label="As-Is (Saat Ini)", color="#2b5c8f")
bar2 = ax1.bar(x + width/2, df["Target_To_Be"], width, label="To-Be (Target)", color="#f28e2b")

ax1.set_title("Perbandingan Capability Level SIAKAD (As-Is vs To-Be)")
ax1.set_ylabel("Skor Capability Level (0 - 5)")
ax1.set_xticks(x)
ax1.set_xticklabels(df["Kode Proses"])
ax1.set_ylim(0, 5.5)
ax1.legend()
ax1.grid(axis="y", linestyle="--", alpha=0.5)
ax1.bar_label(bar1, padding=3)
ax1.bar_label(bar2, padding=3)

# Grafik 2: Peringkat Nilai Gap per Proses (Horizontal Bar)
df_urut = df.sort_values(by="Gap", ascending=True)
warna_gap = ["#d9534f" if g > 1.0 else "#5bc0de" for g in df_urut["Gap"]]
bars_gap = ax2.barh(df_urut["Kode Proses"], df_urut["Gap"], color=warna_gap)

ax2.set_title("Peringkat Nilai Gap Berdasarkan Prioritas Perbaikan")
ax2.set_xlabel("Nilai Gap (To-Be - As-Is)")
ax2.set_xlim(0, 2.2)
ax2.grid(axis="x", linestyle="--", alpha=0.5)
ax2.bar_label(bars_gap, padding=3)

plt.tight_layout()
plt.show()